# H09 — 选择实时推送，而不同时堆所有协议

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H09 — 选择实时推送，而不同时堆所有协议。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H09-realtime.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

状态变化以后，浏览器或另一台设备怎样立即知道？先分别试 WebSocket 和 MQTT，再选择适合自己工具的一条主线。

前置：H08；WebSocket 浏览器，MQTT 可选本地 Mosquitto。

## 本次够用的知识

WebSocket 维护双向连接，适合网页会话；MQTT 借 Broker 解耦发布者/订阅者。retain 是最后状态，不是事件历史；QoS 1 允许重复；订阅要在重连后恢复。接收回调中的 data 可能分片且不以   结束。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 先定义消息语义 · Guided

保留 HTTP 与本地功能。写明哪些是 state、哪些是 event、哪些是 command。示例 topic academy/desk 仅用于一台教学设备，多设备要有唯一 ID。
MQTT 先用自己可信 LAN 中的 Broker，记录地址/端口/配置，别把 localhost 写给 ESP32——那是 ESP32 自己。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 先完成握手和一条消息 · Guided

REQUIRES esp_http_server，menuconfig 启用 CONFIG_HTTPD_WS_SUPPORT。最小示例只 echo，上层主动推送是练习。
浏览器开发者工具 Console：const ws=new WebSocket('ws://<device-ip>/ws'); ws.onmessage=e=>console.log(e.data); ws.onopen=()=>ws.send('hello');。不要从 HTTPS 页面混用 ws://；可信 LAN 用本地 HTTP 页面。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H09/examples/websocket.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_http_server mqtt esp_wifi esp_event esp_netif esp_timer nvs_flash freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "wifi_link.hpp"
#include "esp_http_server.h"
#include "esp_log.h"

static esp_err_t websocket(httpd_req_t* req) {
    if (req->method == HTTP_GET) {
        return ESP_OK; // WebSocket upgrade handshake.
    }
    httpd_ws_frame_t frame{};
    const esp_err_t header_result = httpd_ws_recv_frame(req, &frame, 0);
    if (header_result != ESP_OK) {
        return header_result;
    }
    if (frame.len > 64) {
        return ESP_ERR_INVALID_SIZE;
    }
    uint8_t payload[65]{};
    frame.payload = payload;
    const esp_err_t err = httpd_ws_recv_frame(req, &frame, sizeof(payload) - 1);
    if (err != ESP_OK) {
        return err;
    }
    // Guided echo only; a device-state broadcaster is the learner's challenge.
    return httpd_ws_send_frame(req, &frame);
}
extern "C" void app_main() {
    ESP_ERROR_CHECK(wifi_link_start());
    while (!wifi_link_wait(pdMS_TO_TICKS(1000))) {
    }
    httpd_handle_t server = nullptr;
    httpd_config_t config = HTTPD_DEFAULT_CONFIG();
    ESP_ERROR_CHECK(httpd_start(&server, &config));
    httpd_uri_t uri{};
    uri.uri = "/ws";
    uri.method = HTTP_GET;
    uri.handler = websocket;
    uri.is_websocket = true;
    ESP_ERROR_CHECK(httpd_register_uri_handler(server, &uri));
}


## 通过 Broker 连接另一端 · Guided

REQUIRES mqtt；v5.5.1 基线自带组件，不在这里重复添加别的 esp-mqtt 版本。配置 ACADEMY_MQTT_URI 为电脑的 LAN 地址。
电脑用 mosquitto_sub -h <broker-ip> -t 'academy/desk/#' -v 观察，再 mosquitto_pub -h <broker-ip> -t academy/desk/cmd -m hello。Broker 须监听 LAN 且配置访问边界，服务监听默认 localhost 不会自动供设备访问。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H09/examples/mqtt.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_http_server mqtt esp_wifi esp_event esp_netif esp_timer nvs_flash freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "wifi_link.hpp"
#include "mqtt_client.h"
#include "esp_log.h"
#include "sdkconfig.h"

static void event(void*, esp_event_base_t, int32_t id, void* raw) {
    auto* message = static_cast<esp_mqtt_event_t*>(raw);
    if (id == MQTT_EVENT_CONNECTED) {
        const int subscribed =
            esp_mqtt_client_subscribe(message->client, "academy/desk/cmd", 1);
        const int published = esp_mqtt_client_publish(
            message->client, "academy/desk/status", "online", 0, 1, 1);
        ESP_LOGI("mqtt", "connected subscribe_id=%d publish_id=%d", subscribed,
                 published);
    } else if (id == MQTT_EVENT_DATA) {
        // Fragments are NOT necessarily complete commands; never use strcmp(data,...).
        ESP_LOGI("mqtt", "chunk=%d offset=%d total=%d topic=%.*s data=%.*s",
                 message->data_len, message->current_data_offset,
                 message->total_data_len, message->topic_len,
                 message->topic ? message->topic : "", message->data_len,
                 message->data);
    } else if (id == MQTT_EVENT_DISCONNECTED) {
        ESP_LOGW("mqtt", "broker disconnected; client will reconnect");
    }
}
extern "C" void app_main() {
    ESP_ERROR_CHECK(wifi_link_start());
    while (!wifi_link_wait(pdMS_TO_TICKS(1000))) {
    }
    esp_mqtt_client_config_t cfg{};
    cfg.broker.address.uri = CONFIG_ACADEMY_MQTT_URI;
    cfg.session.last_will.topic = "academy/desk/status";
    cfg.session.last_will.msg = "offline";
    cfg.session.last_will.qos = 1;
    cfg.session.last_will.retain = 1;
    esp_mqtt_client_handle_t client = esp_mqtt_client_init(&cfg);
    if (!client) {
        ESP_LOGE("mqtt", "client allocation failed");
        return;
    }
    ESP_ERROR_CHECK(
        esp_mqtt_client_register_event(client, MQTT_EVENT_ANY, event, nullptr));
    ESP_ERROR_CHECK(esp_mqtt_client_start(client));
}


## 状态与事件分开 · Modify

把连接状态 retain 到 status，按钮按下事件设 retain=false。重启订阅客户端，对比哪些会重放。
停 Broker 再启动，观察 LWT、重连和订阅。QoS 1 下不能假设一个发布永远只消费一次，业务 command 应可幂等或带 ID。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 旧状态不是新动作 · Debug

临时把“启动计时”事件设 retain=true，新客户端/设备重连观察旧命令风险，立即恢复并清除测试 retained command。
MQTT 分片测试发送超过接收缓冲的 payload，记录 offset/total_data_len；不把每块都执行为一条命令。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 完成一种实时通道 · Build

任选 WebSocket 状态广播或 MQTT 工具。要求：消息版本与大小有界；断连本地功能继续；重连恢复；慢客户端/队列满有策略；完整 payload 校验后才提交事件。
WebSocket 网络操作放 server 工作队列，连接失效可恢复；MQTT 为多设备设独立 topic，处理重复 command。写断线/重复/分片测试。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 解释选择 · Open

写出为什么自己的需求选择 HTTP polling、WebSocket 或 MQTT；设备没有网页客户端时是否仍需要 WebSocket？
H10 再根据真实负载决定 Task，而不是每种协议创建一个 Task。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] 至少一种实时通道完成真实电脑↔设备往返
- [ ] 重连与订阅/会话恢复有故障记录
- [ ] retain、重复与分片语义有测试

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H09',
    [
        '至少一种实时通道完成真实电脑↔设备往返',
        '重连与订阅/会话恢复有故障记录',
        'retain、重复与分片语义有测试',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H09/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/protocols/esp_http_server.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/protocols/esp_http_server.html)
- [api-reference/protocols/mqtt.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/protocols/mqtt.html)